# 05_KMeans_nivel2


In [1]:
import pandas as pd
import numpy as np
import time, joblib
from pathlib import Path
from datetime import datetime
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase4_gustos' else Path.cwd()
DATA_QC = ROOT / 'data' / 'data_qc_gustos'
DATA_MODELS = ROOT / 'data' / 'data_models_gustos'
INFORMES_BASE = ROOT / 'informes' / 'fase4_gustos' / '04_clustering' / 'nivel2'
DATA_MODELS.mkdir(parents=True, exist_ok=True)

MASTER_PATH = DATA_QC / 'features_tier2.parquet'
TIER_LEVEL = '2'

def reduce_high_card(df, max_unique=10):
    out = df.copy()
    for cat in out.select_dtypes(include=['object','category']).columns.tolist():
        if out[cat].nunique(dropna=True) > max_unique:
            top = out[cat].value_counts().head(max_unique).index.tolist()
            out[cat] = out[cat].where(out[cat].isin(top), 'OTHER')
    return out

def preprocess(df, nan_threshold_zero=0.30):
    df = df.drop(columns=['user_id']).copy()
    numeric = df.select_dtypes(include=['number']).columns.tolist()
    categorical = df.select_dtypes(include=['object','category']).columns.tolist()
    nan_pcts = df[numeric].isna().mean()
    num_low = nan_pcts[nan_pcts < nan_threshold_zero].index.tolist()
    num_high = nan_pcts[nan_pcts >= nan_threshold_zero].index.tolist()
    df = reduce_high_card(df, 10)
    pp = ColumnTransformer([
        ('num_low', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', RobustScaler())]), num_low),
        ('num_high', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value=0)), ('sc', RobustScaler())]), num_high),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical),
    ])
    X = pp.fit_transform(df)
    return X, pp, pp.get_feature_names_out()

print(f'TIER {TIER_LEVEL}: cargando {MASTER_PATH.name}')
master = pd.read_parquet(MASTER_PATH)
user_ids = master['user_id'].values
print(f'  shape: {master.shape}')
X, preproc, names = preprocess(master)
joblib.dump(preproc, DATA_MODELS / f'preprocessor_tier{TIER_LEVEL}.joblib')
print(f'  X post-preproc: {X.shape}')
N = len(X)

from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
REPORT = INFORMES_BASE / '05a_kmeans'
REPORT.mkdir(parents=True, exist_ok=True)
K_RANGE = list(range(2, 11))
SAMPLE_SIL = 20000
results = []


TIER 2: cargando features_tier2.parquet
  shape: (21599, 17)
  X post-preproc: (21599, 16)


In [2]:
rng = np.random.RandomState(42)
sil_idx = rng.choice(len(X), min(SAMPLE_SIL, len(X)), replace=False)
X_sil = X[sil_idx]
inertias=[]; sils=[]
for k in K_RANGE:
    t0 = time.time()
    if len(X) > 50_000:
        km = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=2048, n_init=3)
    else:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    labels_sub = km.predict(X_sil)
    sil = float(silhouette_score(X_sil, labels_sub)) if len(set(labels_sub))>1 else float('nan')
    db = float(davies_bouldin_score(X, labels))
    ch = float(calinski_harabasz_score(X, labels))
    unique, counts = np.unique(labels, return_counts=True)
    max_pct = float(counts.max()/len(labels))
    min_pct = float(counts.min()/len(labels))
    inertias.append(km.inertia_); sils.append(sil)
    results.append({'algorithm':'KMeans','tier':TIER_LEVEL,'k':k,'silhouette':sil,'davies_bouldin':db,'calinski_harabasz':ch,'inertia':float(km.inertia_),'n_clusters_actual':int(len(set(labels))),'max_cluster_pct':max_pct,'min_cluster_pct':min_pct,'elapsed_s':time.time()-t0})
    print(f'  K={k}: sil={sil:.4f} db={db:.4f} max%={max_pct:.1%} min%={min_pct:.1%}')
    joblib.dump({'model':km,'labels':labels,'user_ids':user_ids}, DATA_MODELS / f'nivel{TIER_LEVEL}_kmeans_k{k}.joblib')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_RANGE, inertias, marker='o'); axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia'); axes[0].grid(alpha=0.3); axes[0].set_title(f'Elbow nivel{TIER_LEVEL}')
axes[1].plot(K_RANGE, sils, marker='o', color='C1'); axes[1].axhline(0.15, color='red', ls='--', alpha=0.4); axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette'); axes[1].grid(alpha=0.3); axes[1].set_title(f'Silhouette nivel{TIER_LEVEL}')
plt.tight_layout(); plt.savefig(REPORT / 'elbow_silhouette.png', dpi=120, bbox_inches='tight'); plt.close()
pd.DataFrame(results).to_csv(REPORT / 'metrics.csv', index=False)
print('OK metrics.csv guardado')


  K=2: sil=0.9805 db=0.6056 max%=99.7% min%=0.3%


  K=3: sil=0.9782 db=0.4049 max%=99.6% min%=0.0%


  K=4: sil=0.9674 db=0.5651 max%=99.4% min%=0.0%


  K=5: sil=0.9210 db=0.6505 max%=98.3% min%=0.0%


  K=6: sil=0.9245 db=0.6304 max%=98.3% min%=0.0%


  K=7: sil=0.9166 db=0.6552 max%=98.2% min%=0.0%


  K=8: sil=0.9130 db=0.5982 max%=98.1% min%=0.0%


  K=9: sil=0.8703 db=0.6929 max%=97.2% min%=0.0%


  K=10: sil=0.8425 db=0.6712 max%=96.7% min%=0.0%
OK metrics.csv guardado
